# Data Quality & Sanity Checks: AppointmentRecords

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

df = pd.read_csv("../data/AppointmentRecords.csv", parse_dates=["AppointmentDateTime"])
df["Date"] = df["AppointmentDateTime"].dt.normalize()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   AppointmentID         129353 non-null  int64         
 1   ClinicID              129353 non-null  int64         
 2   ClinicName            129353 non-null  str           
 3   ClinicType            129353 non-null  str           
 4   PatientID             129353 non-null  int64         
 5   ProviderID            129353 non-null  int64         
 6   AppointmentDateTime   129353 non-null  datetime64[us]
 7   AppointmentType       129353 non-null  str           
 8   Status                129353 non-null  str           
 9   BookingChannel        129353 non-null  str           
 10  BookingLeadTimeDays   129353 non-null  int64         
 11  IsHoliday             129353 non-null  bool          
 12  HolidayName           939 non-null     str           
 13  SeasonFlag

In [3]:
print(df.isna().sum()[lambda s: s > 0])  # HolidayName is nullable by design
print("Duplicate AppointmentID:", df["AppointmentID"].duplicated().sum())
print("Duplicate rows (ignoring ID):", df.drop(columns="AppointmentID").duplicated().sum())

HolidayName    128414
dtype: int64
Duplicate AppointmentID: 0
Duplicate rows (ignoring ID): 1


Booking lead time = number of days between booking and the appointment time so it cannot be negative

In [ ]:
allowed = {
    "ClinicType": {"Primary Care", "Multi-Specialty", "Telehealth-Forward"},
    "AppointmentType": {"New Patient", "Follow-Up", "Telehealth", "Urgent"},
    "Status": {"Completed", "No-Show", "Cancelled", "Rescheduled"},
    "BookingChannel": {"Online", "Phone", "Walk-In", "Referral"},
    "SeasonFlag": {"Flu Season", "Allergy Season", "Standard"},
}

for col, values in allowed.items():
    print(f"Invalid {col}:", (~df[col].isin(values)).sum())

print("Negative lead time:", (df["BookingLeadTimeDays"] < 0).sum())
print(df["AppointmentDateTime"].min(), "->", df["AppointmentDateTime"].max())

Invalid ClinicType: 0
Invalid AppointmentType: 0
Invalid Status: 0
Invalid BookingChannel: 0
Invalid SeasonFlag: 0
Negative lead time: 0
2024-01-01 08:00:00 -> 2025-12-31 17:45:00


Understand what happens (season label) each month gets

In [7]:
pd.crosstab(df["AppointmentDateTime"].dt.month, df["SeasonFlag"])

SeasonFlag,Allergy Season,Flu Season,Standard
AppointmentDateTime,,,
1,0,10939,0
2,0,10341,0
3,9872,0,0
4,10640,0,0
5,10212,0,0
6,0,0,8796
7,0,0,10887
8,0,0,11333
9,0,0,10934


In [9]:
daily = df.groupby(["ClinicName", "Date"]).size().unstack(0)  # NaN = no appointments that day
coverage = pd.DataFrame({
    "first_date": daily.apply(pd.Series.first_valid_index),
    "last_date": daily.apply(pd.Series.last_valid_index),
    "zero_days": daily.isna().sum(),
    "zero_sundays": daily[daily.index.dayofweek == 6].isna().sum(),
})
coverage

,first_date,last_date,zero_days,zero_sundays
ClinicName,,,,
Nexora Care - Downtown Express,2024-01-01,2025-12-31,0,0
Nexora Care - Hillcrest,2024-06-29,2025-12-31,259,104
Nexora Care - Lakeside,2024-01-01,2025-12-31,104,104
Nexora Care - Riverside,2024-01-01,2025-12-31,104,104


Sanity check is the minimal acceptable quality of our data before analysis

In [ ]:
# sanity checks
assert df["AppointmentID"].is_unique
assert df["AppointmentDateTime"].notna().all()
assert df.drop(columns="HolidayName").notna().all().all()
assert all(df[c].isin(v).all() for c, v in allowed.items())
assert (df["BookingLeadTimeDays"] >= 0).all()
print("All sanity tests passed")

All sanity tests passed
